In [0]:
dbutils.widgets.removeAll()

In [0]:
# /Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/FactGapsInCare/FactGapsInCare/caregap_orchestrator

dbutils.widgets.removeAll()
import os
import sys
import shutil
from pathlib import Path
import pandas as pd

os.sync()

ROOT_DIR = Path(
    "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing"
)
sys.path.append(str(ROOT_DIR))

%run ./caregap_analyzer

from Shared.EDIProcessing import EDIProcessor, CSVConverter
from DimMember.EDIProcessing.mapper import Mapper
from FactGapsInCare.EDIProcessing.mapper import Mapper as ClaimsMapper


def move_file(src_path: Path, target_dir: Path) -> Path:
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    shutil.move(str(src_path), str(target_path))
    return target_path


def normalize_dataframe(df: pd.DataFrame, file_type: str) -> pd.DataFrame:
    if df.empty:
        return df

    df.columns = df.columns.str.strip().str.lower()

    if file_type == "834":
        if "dateofbirth" in df.columns and "dob" not in df.columns:
            df["dob"] = df["dateofbirth"]

        if "uniquepersonkey" in df.columns and "member_id" not in df.columns:
            df["member_id"] = df["uniquepersonkey"]
        elif "beneficiaryid" in df.columns and "member_id" not in df.columns:
            df["member_id"] = df["beneficiaryid"]

        if (
            "eligibilitystartdate" in df.columns
            and "enrollment_start" not in df.columns
        ):
            df["enrollment_start"] = df["eligibilitystartdate"]

    elif file_type == "837":
        if "patient_dob" in df.columns and "dob" not in df.columns:
            df["dob"] = df["patient_dob"]
        if "patient_id" in df.columns and "member_id" not in df.columns:
            df["member_id"] = df["patient_id"]

    return df


def process_edi_file(
    file_path: Path, base_source_dir: Path, file_type: str
) -> tuple:
    active_file_path = file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")

        active_file_path = move_file(
            active_file_path, base_source_dir / "inprogress"
        )
        structured_json = EDIProcessor().parse(str(active_file_path))

        if file_type == "834":
            mapped_records = Mapper().map_member(structured_json)
            layout_id = "834"
        else:
            mapped_records = ClaimsMapper().map_claims(structured_json)
            layout_id = "837"

        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            CSVConverter().converter(mapped_records, str(target_csv_path))
        except Exception as csv_error:
            print(f"Warning: CSV conversion failed: {csv_error}")

        if isinstance(mapped_records, list):
            df = pd.DataFrame(mapped_records)
        elif isinstance(mapped_records, dict):
            df = pd.DataFrame([mapped_records])
        elif target_csv_path.exists():
            df = pd.read_csv(target_csv_path)
        else:
            raise ValueError(
                f"Cannot convert mapped_records to DataFrame. Type: {type(mapped_records)}"
            )

        print(
            f"[{file_type}] Processed {active_file_path.name} -> CSV created at {target_csv_path}"
        )
        return df, active_file_path

    except Exception as e:
        print(
            f"Failed processing {file_type} file {active_file_path.name}: {e}"
        )
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise


def finalize_file_tracking(
    file_path: Path, base_source_dir: Path, success: bool
):
    try:
        if success:
            print(f"--> Archiving raw file to processed: {file_path.name}")
            move_file(file_path, base_source_dir / "processed")
        else:
            print(f"--> Moving raw file to failed: {file_path.name}")
            move_file(file_path, base_source_dir / "failed")
    except Exception as e:
        print(f"Failed to update tracking directory state: {e}")


def get_pending_files(source_dir: Path) -> list:
    pending_dir = source_dir / "pending"
    if not pending_dir.exists():
        return []
    return [
        f
        for f in pending_dir.iterdir()
        if f.is_file() and not f.name.startswith(".")
    ]


def main():
    os.sync()

    base_834_dir = ROOT_DIR / "source/834"
    base_837_dir = ROOT_DIR / "source/837"

    pending_834_files = get_pending_files(base_834_dir)
    pending_837_files = get_pending_files(base_837_dir)

    print(f"DEBUG: pending_834_files = {pending_834_files}")
    print(f"DEBUG: pending_837_files = {pending_837_files}")

    if not pending_834_files and not pending_837_files:
        print("No pending 834 or 837 files found to process.")
        return

    enrollment_dfs = []
    processed_834_trackers = []
    for file_path in pending_834_files:
        try:
            df, tracked_file = process_edi_file(
                file_path, base_834_dir, file_type="834"
            )
            enrollment_dfs.append(df)
            processed_834_trackers.append((tracked_file, base_834_dir))
        except Exception as e:
            print(f"Skipping failed 834 file: {file_path.name}")

    claims_dfs = []
    processed_837_trackers = []
    for file_path in pending_837_files:
        try:
            df, tracked_file = process_edi_file(
                file_path, base_837_dir, file_type="837"
            )
            claims_dfs.append(df)
            processed_837_trackers.append((tracked_file, base_837_dir))
        except Exception as e:
            print(f"Skipping failed 837 file: {file_path.name}")

    enrollment_df = (
        pd.concat(enrollment_dfs, ignore_index=True)
        if enrollment_dfs
        else pd.DataFrame()
    )
    claims_df = (
        pd.concat(claims_dfs, ignore_index=True)
        if claims_dfs
        else pd.DataFrame()
    )

    enrollment_df = normalize_dataframe(enrollment_df, "834")
    claims_df = normalize_dataframe(claims_df, "837")

    if enrollment_df.empty:
        print("No enrollment data available to calculate care gaps.")
        for tracked_file, base_dir in (
            processed_834_trackers + processed_837_trackers
        ):
            finalize_file_tracking(tracked_file, base_dir, success=True)
        return

    lookup_df = spark.table(
        "claimsprocessing.bronze.measure_library"
    ).toPandas()

    try:
        print("\n=== Executing calculate_care_gaps ===")
        results_df = calculate_care_gaps(
            member_df=enrollment_df,
            claims_df=claims_df,
            lookup_df=lookup_df,
            measurement_year=2026,
        )

        print(
            f"Care Gap Calculation Complete. Evaluated {len(results_df)} records."
        )
        display(results_df)

    except Exception as e:
        print(f"Care gap processing failed: {e}")

    print("\n=== Archiving processed files ===")
    for tracked_file, base_dir in (
        processed_834_trackers + processed_837_trackers
    ):
        finalize_file_tracking(tracked_file, base_dir, success=True)


if __name__ == "__main__":
    main()

In [0]:
import os
import sys
import shutil
from pathlib import Path
import json
import pandas as pd

In [0]:
# Force local file system synchronization
os.sync()

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
%run ./caregap_analyzer

In [0]:
from Shared.EDIProcessing import EDIProcessor, CSVConverter
from DimMember.EDIProcessing.mapper import Mapper
from FactGapsInCare.EDIProcessing.mapper import Mapper as ClaimsMapper

In [0]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    shutil.move(str(src_path), str(target_path))
    return target_path

In [0]:
def normalize_dataframe(df: pd.DataFrame, file_type: str) -> pd.DataFrame:
    """Standardizes column names for calculate_hedis_care_gaps."""
    if df.empty:
        return df

    # Lowercase all headers to avoid casing mismatches
    df.columns = df.columns.str.strip().str.lower()

    if file_type == "834":
        # Map birth date field to 'dob'
        if "dateofbirth" in df.columns and "dob" not in df.columns:
            df["dob"] = df["dateofbirth"]

        # Map member identifier to 'member_id'
        if "uniquepersonkey" in df.columns and "member_id" not in df.columns:
            df["member_id"] = df["uniquepersonkey"]
        elif "beneficiaryid" in df.columns and "member_id" not in df.columns:
            df["member_id"] = df["beneficiaryid"]

    elif file_type == "837":
        if "patient_dob" in df.columns and "dob" not in df.columns:
            df["dob"] = df["patient_dob"]
        if "patient_id" in df.columns and "member_id" not in df.columns:
            df["member_id"] = df["patient_id"]

    return df

In [0]:
def process_edi_file(file_path: Path, base_source_dir: Path, file_type: str) -> tuple:
    """Parses 834 or 837 file, maps records, writes temp CSV, and returns Pandas DataFrame."""
    active_file_path = file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        # Move raw file to inprogress
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")

        # Parse EDI JSON
        structured_json = EDIProcessor().parse(str(active_file_path))
        
        # Map based on file type (834 Member vs 837 Claims)
        if file_type == "834":
            mapped_records = Mapper().map_member(structured_json)
            layout_id = "834"
        else:
            mapped_records = ClaimsMapper().map_claims(structured_json)
            layout_id = "837"

        # Temp CSV File Generation
        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Try to write CSV for archival purposes (optional)
        try:
            CSVConverter().converter(mapped_records, str(target_csv_path))
        except Exception as csv_error:
            print(f"Warning: CSV conversion failed: {csv_error}")
        
        # Convert mapped data to DataFrame directly from mapped_records
        # This is more reliable than writing and reading CSV
        if isinstance(mapped_records, list):
            df = pd.DataFrame(mapped_records)
        elif isinstance(mapped_records, dict):
            # If single dict, wrap in list
            df = pd.DataFrame([mapped_records])
        elif target_csv_path.exists():
            # Fallback: read from CSV if it was successfully created
            df = pd.read_csv(target_csv_path)
        else:
            raise ValueError(f"Cannot convert mapped_records to DataFrame. Type: {type(mapped_records)}")

        print(f"[{file_type}] Processed {active_file_path.name} -> CSV created at {target_csv_path}")
        return df, active_file_path

    except Exception as e:
        print(f"Failed processing {file_type} file {active_file_path.name}: {e}")
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise

In [0]:
def finalize_file_tracking(file_path: Path, base_source_dir: Path, success: bool):
    """Moves the raw EDI file based on lifecycle completion status."""
    try:
        if success:
            print(f"--> Archiving raw file to processed: {file_path.name}")
            move_file(file_path, base_source_dir / "processed")
        else:
            print(f"--> Moving raw file to failed: {file_path.name}")
            move_file(file_path, base_source_dir / "failed")
    except Exception as e:
        print(f"Failed to update tracking directory state: {e}")

In [0]:
def get_pending_files(source_dir: Path) -> list:
    """Returns list of non-hidden files in the pending directory."""
    pending_dir = source_dir / "pending"
    if not pending_dir.exists():
        return []
    return [f for f in pending_dir.iterdir() if f.is_file() and not f.name.startswith('.')]

In [0]:
def main():
    # Force filesystem sync to see newly added files
    os.sync()
    
    base_834_dir = ROOT_DIR / "source/834"
    base_837_dir = ROOT_DIR / "source/837"

    pending_834_files = get_pending_files(base_834_dir)
    pending_837_files = get_pending_files(base_837_dir)
    
    print(f"DEBUG: base_837_dir = {base_837_dir}")
    print(f"DEBUG: pending_837_files = {pending_837_files}")

    if not pending_834_files and not pending_837_files:
        print("No pending 834 or 837 files found to process.")
        return

    # 1. Process 834 Pending Files (Enrollment)
    enrollment_dfs = []
    processed_834_trackers = []

    for file_path in pending_834_files:
        try:
            df, tracked_file = process_edi_file(file_path, base_834_dir, file_type="834")
            enrollment_dfs.append(df)
            processed_834_trackers.append((tracked_file, base_834_dir))
        except Exception as e:
            print(f"Skipping failed 834 file: {file_path.name}")

    # 2. Process 837 Pending Files (Claims)
    claims_dfs = []
    processed_837_trackers = []

    for file_path in pending_837_files:
        try:
            df, tracked_file = process_edi_file(file_path, base_837_dir, file_type="837")
            claims_dfs.append(df)
            processed_837_trackers.append((tracked_file, base_837_dir))
        except Exception as e:
            print(f"Skipping failed 837 file: {file_path.name}")

    # Combine into unified DataFrames
    enrollment_df = pd.concat(enrollment_dfs, ignore_index=True) if enrollment_dfs else pd.DataFrame()
    claims_df = pd.concat(claims_dfs, ignore_index=True) if claims_dfs else pd.DataFrame()

    enrollment_df = normalize_dataframe(enrollment_df, "834")
    claims_df = normalize_dataframe(claims_df, "837")
    
    if enrollment_df.empty:
        print("No enrollment data available to calculate care gaps.")
        # Archive all successfully parsed files before returning
        for tracked_file, base_dir in processed_834_trackers + processed_837_trackers:
            finalize_file_tracking(tracked_file, base_dir, success=True)
        return

    # 3. Read HEDIS measure lookup table from catalog
    lookup_df = spark.table("claimsprocessing.bronze.measure_library").toPandas()

    # 4. Execute Care Gap Calculation
    try:
        print("\n=== Executing calculate_hedis_care_gaps ===")
        results_df = calculate_hedis_care_gaps(
            enrollment_df=enrollment_df,
            claims_df=claims_df,
            lookup_df=lookup_df,
            measurement_year=2026
        )

        print(f"Care Gap Calculation Complete. Evaluated {len(results_df)} records.")
        display(results_df)

    except Exception as e:
        print(f"Care gap processing failed: {e}")
    
    # Archive all successfully parsed files at the end
    print("\n=== Archiving processed files ===")
    for tracked_file, base_dir in processed_834_trackers + processed_837_trackers:
        finalize_file_tracking(tracked_file, base_dir, success=True)

In [0]:
if __name__ == "__main__":
    main()